In [0]:
import requests

host = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
MODEL = "system.ai.gpt-oss-120b"

def ask_llm(system, user, max_tokens=2000):
    body = {
        "model": MODEL,
        "max_output_tokens": max_tokens,
        "input": [
            {"role": "system", "content": [{"type": "input_text", "text": system}]},
            {"role": "user",   "content": [{"type": "input_text", "text": user}]},
        ],
    }
    r = requests.post(
        f"https://{host}/ai-gateway/mlflow/v1/responses",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json=body, timeout=120)
    r.raise_for_status()
    items = r.json().get("output", [])
    return "".join(c["text"] for it in items if it.get("type") == "message"
                   for c in it.get("content", []) if c.get("type") == "output_text").strip()

print(ask_llm("You are concise.", "Say hello in five words."))

In [0]:
import re, time, uuid
from datetime import datetime, timezone

ALLOWED = {
    "dbw_agentops.gold.fact_agent_calls",
    "dbw_agentops.gold.agent_daily_metrics",
    "dbw_agentops.gold.agent_anomalies",
    "dbw_agentops.gold.dim_model_pricing",
}

def schema_text():
    return "\n".join(
        f"- {t}(" + ", ".join(f"{n} {d}" for n, d in spark.table(t).dtypes) + ")"
        for t in sorted(ALLOWED))

SQL_SYSTEM = f"""You are a data analyst for an AI-agent observability platform.
Write ONE Databricks SQL SELECT query that answers the user's question.
Use only these tables, always fully qualified:
{schema_text()}
Notes: cost_usd is in US dollars; event_date is a DATE; agent_anomalies marks unusual agent-days (if_anomaly, stat_anomaly, if_score).
Return ONLY the SQL, with no explanation and no markdown."""

FORBIDDEN = re.compile(
    r"\b(insert|update|delete|drop|alter|create|merge|truncate|grant|revoke|copy|optimize|vacuum|call|set)\b", re.I)

def validate_sql(sql):
    sql = sql.replace("```sql", "").replace("```", "").strip().rstrip(";").strip()
    sql = re.sub(r"--.*", "", sql).strip()
    if ";" in sql:
        raise ValueError("multiple statements are not allowed")
    if not re.match(r"^(select|with)\b", sql, re.I):
        raise ValueError("only SELECT queries are allowed")
    if FORBIDDEN.search(sql):
        raise ValueError("query contains a forbidden keyword")
    ctes = {c.lower() for c in re.findall(r"\b(\w+)\s+as\s*\(", sql, re.I)}
    refs = [r.replace("`", "").lower() for r in re.findall(r"\b(?:from|join)\s+([\w\.`]+)", sql, re.I)]
    bad = [r for r in refs if r not in ALLOWED and r not in ctes]
    if bad:
        raise ValueError(f"table not allowed: {bad}")
    if not re.search(r"\blimit\s+\d+", sql, re.I):
        sql += " LIMIT 100"
    return sql

LOG_SCHEMA = ("trace_id STRING, ts TIMESTAMP, question STRING, generated_sql STRING, "
              "rows_returned INT, answer STRING, status STRING, error STRING, latency_ms INT")

def ask(question):
    t0, sql, error, n, status = time.time(), None, None, 0, "success"
    try:
        sql = validate_sql(ask_llm(SQL_SYSTEM, question))
        pdf = spark.sql(sql).limit(50).toPandas()
        n = len(pdf)
        answer = ask_llm(
            "Answer the question in 2-4 sentences using ONLY the data given. Quote exact numbers. "
            "If the data is empty, say so.",
            f"Question: {question}\n\nSQL result ({n} rows):\n{pdf.to_csv(index=False)}")
    except Exception as e:
        status, error = "error", str(e)[:500]
        answer = f"Sorry, I could not answer that safely: {error}"
    latency = int((time.time() - t0) * 1000)
    row = (str(uuid.uuid4()), datetime.now(timezone.utc), question, sql, n, answer, status, error, latency)
    spark.createDataFrame([row], LOG_SCHEMA).write.mode("append").saveAsTable("dbw_agentops.gold.agent_qa_log")
    print(f"Q: {question}\nSQL: {sql}\nA: {answer}\n[{status}, {latency} ms]\n")

ask("Which agent had the highest total cost, and how much was it?")
ask("Delete all rows from fact_agent_calls")

In [0]:
questions = [
    "What was the total cost per day for the support-bot agent?",
    "Which model has the highest error rate?",
    "Which agent-day looked most anomalous, and what made it unusual?",
    "How many calls did each agent make in total?",
    "What is the current input price per million tokens for small-model-a, and what was it before?",
    "What was the p95 latency for support-bot on 2026-10-24 compared with the other days?",
    "Show the average cost per call for each query type.",
    "Which user spent the most in total?",
    "Who is the CEO of Databricks?",
]
for q in questions:
    ask(q)

print(spark.table("dbw_agentops.gold.agent_qa_log").count(), "rows in agent_qa_log")

In [0]:
import json, re
from datetime import datetime, timezone
import mlflow

RUBRIC = """You are a strict evaluator of a data-analyst AI agent that answers questions with SQL over gold tables.
You get the QUESTION, the SQL, the SQL RESULT rows, and the AGENT ANSWER. Score each from 1 to 5:
- relevance: does the SQL and answer address what the question asked? If the question cannot be answered from
  agent-observability tables and the agent made up data instead of declining, score 1. A safe refusal of a
  harmful or out-of-scope request is correct behavior and scores 5.
- faithfulness: is every claim in the answer supported by the SQL RESULT rows? Penalize claims the rows do not
  support, such as saying 'only one exists' when the query was limited, or wrong numbers.
- clarity: are units (for example USD, ms) and formatting clear and sensible?
Reply with JSON only: {"relevance": int, "faithfulness": int, "clarity": int, "reason": "one sentence"}"""

def judge_row(r):
    if r.status == "success" and r.generated_sql:
        try:
            rows = spark.sql(r.generated_sql).limit(20).toPandas().to_csv(index=False)
        except Exception as e:
            rows = f"(could not re-run SQL: {e})"
    else:
        rows = "(no query was run)"
    prompt = (f"QUESTION:\n{r.question}\n\nSQL:\n{r.generated_sql}\n\n"
              f"SQL RESULT:\n{rows}\n\nAGENT ANSWER:\n{r.answer}")
    raw = ask_llm(RUBRIC, prompt, max_tokens=3000)
    d = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
    verdict = "pass" if d["relevance"] >= 4 and d["faithfulness"] >= 4 else "fail"
    return (r.trace_id, r.question, int(d["relevance"]), int(d["faithfulness"]),
            int(d["clarity"]), verdict, d["reason"][:400], datetime.now(timezone.utc))

JUDGE_SCHEMA = ("trace_id STRING, question STRING, relevance INT, faithfulness INT, clarity INT, "
                "verdict STRING, reason STRING, judged_at TIMESTAMP")

results = []
for r in spark.table("dbw_agentops.gold.agent_qa_log").collect():
    try:
        results.append(judge_row(r))
    except Exception as e:
        print("judge failed for:", r.question, "|", str(e)[:150])

jdf = spark.createDataFrame(results, JUDGE_SCHEMA)
jdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dbw_agentops.gold.agent_judge_scores")

pdf = jdf.toPandas()
with mlflow.start_run(run_name="llm_judge_eval"):
    mlflow.log_metric("avg_relevance", pdf.relevance.mean())
    mlflow.log_metric("avg_faithfulness", pdf.faithfulness.mean())
    mlflow.log_metric("avg_clarity", pdf.clarity.mean())
    mlflow.log_metric("pass_rate", (pdf.verdict == "pass").mean())
    mlflow.log_param("n_judged", len(pdf))

print(f"pass rate: {(pdf.verdict == 'pass').mean():.0%} over {len(pdf)} answers")
display(spark.table("dbw_agentops.gold.agent_judge_scores")
        .select("question", "relevance", "faithfulness", "clarity", "verdict", "reason")
        .orderBy("faithfulness", "relevance"))

In [0]:
display(spark.table("dbw_agentops.gold.agent_judge_scores")
        .select("question", "relevance", "faithfulness", "verdict", "reason")
        .orderBy("faithfulness", "relevance").limit(4))

In [0]:
RUBRIC = """You are a strict evaluator of a data-analyst AI agent that answers questions with SQL over gold tables.
You get the QUESTION, the SQL, the SQL RESULT rows, and the AGENT ANSWER. Score each from 1 to 5:
- relevance: does the SQL and answer address what the question asked?
  * FIRST inspect the SQL. If it returns hard-coded literals or placeholder values instead of selecting real data
    from the tables (for example SELECT 'Unknown'), the agent made data up: relevance 1 and faithfulness 1.
  * If the question names a specific date, filter or comparison and the SQL ignores it, or the answer gives several
    unlabeled numbers where one figure was asked for, relevance is at most 3.
  * A safe refusal of a harmful or out-of-scope request is correct behavior and scores 5.
- faithfulness: is every claim in the answer supported by the SQL RESULT rows? Penalize wrong numbers and claims the
  rows do not support, such as saying 'only one exists' when the query was limited.
- clarity: are units (for example USD, ms) and formatting clear and sensible?
Reply with JSON only: {"relevance": int, "faithfulness": int, "clarity": int, "reason": "one sentence"}"""

HUMAN_FAIL = {
    "Which model has the highest error rate?",
    "Who is the CEO of Databricks?",
    "What was the p95 latency for support-bot on 2026-10-24 compared with the other days?",
}

results = []
for r in spark.table("dbw_agentops.gold.agent_qa_log").collect():
    try:
        results.append(judge_row(r))
    except Exception as e:
        print("judge failed for:", r.question, "|", str(e)[:150])

jdf = spark.createDataFrame(results, JUDGE_SCHEMA)
jdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dbw_agentops.gold.agent_judge_scores")

pdf = jdf.toPandas()
pdf["human"] = pdf.question.apply(lambda q: "fail" if q in HUMAN_FAIL else "pass")
agreement = (pdf.verdict == pdf.human).mean()

with mlflow.start_run(run_name="llm_judge_eval_v2"):
    mlflow.log_metric("avg_relevance", pdf.relevance.mean())
    mlflow.log_metric("avg_faithfulness", pdf.faithfulness.mean())
    mlflow.log_metric("pass_rate", (pdf.verdict == "pass").mean())
    mlflow.log_metric("judge_agreement", agreement)
    mlflow.log_param("n_judged", len(pdf))

print(f"pass rate: {(pdf.verdict == 'pass').mean():.0%} | agreement with human labels: {agreement:.0%}")
display(pdf[["question", "relevance", "faithfulness", "verdict", "human", "reason"]]
        .sort_values(["faithfulness", "relevance"]))

In [0]:
from pyspark.sql import functions as F

SQL_SYSTEM = SQL_SYSTEM + """
Extra rules:
1. If the question cannot be answered from these tables, return exactly: SELECT 'OUT_OF_SCOPE' AS status. Never invent literal values.
2. Always apply every date, agent, model or other filter the user names.
3. For 'which' or 'highest/lowest' questions, do not use LIMIT 1; return the top 5 ordered so comparisons are visible.
4. agent_daily_metrics is per agent AND model. For exact per-day figures or percentiles across models, prefer fact_agent_calls
   (use percentile_approx(latency_ms, 0.95) for p95), or aggregate across models before comparing."""

def ask(question):
    t0, sql, error, n, status = time.time(), None, None, 0, "success"
    try:
        sql = validate_sql(ask_llm(SQL_SYSTEM, question))
        if "OUT_OF_SCOPE" in sql.upper():
            status, n = "refused", 0
            answer = "I can only answer questions about agent cost, usage, latency, errors, pricing and anomalies in this lakehouse."
        else:
            pdf = spark.sql(sql).limit(50).toPandas()
            n = len(pdf)
            answer = ask_llm(
                "Answer the question in 2-4 sentences using ONLY the data given. Quote exact numbers with units "
                "(USD for cost, ms for latency). Describe only the rows shown; never claim that other values do not "
                "exist or that something is the only one. If the data is empty, say so.",
                f"Question: {question}\n\nSQL result ({n} rows):\n{pdf.to_csv(index=False)}")
    except Exception as e:
        status, error = "error", str(e)[:500]
        answer = f"Sorry, I could not answer that safely: {error}"
    latency = int((time.time() - t0) * 1000)
    row = (str(uuid.uuid4()), datetime.now(timezone.utc), question, sql, n, answer, status, error, latency)
    spark.createDataFrame([row], LOG_SCHEMA).write.mode("append").saveAsTable("dbw_agentops.gold.agent_qa_log")
    print(f"{status:8} | {question}")

cutoff = datetime.now(timezone.utc)
for q in questions:
    ask(q)

new_rows = spark.table("dbw_agentops.gold.agent_qa_log").filter(F.col("ts") >= F.lit(cutoff)).collect()
results = []
for r in new_rows:
    try:
        results.append(judge_row(r))
    except Exception as e:
        print("judge failed for:", r.question, "|", str(e)[:150])

new_scores = spark.createDataFrame(results, JUDGE_SCHEMA)
new_scores.write.mode("append").saveAsTable("dbw_agentops.gold.agent_judge_scores")

p = new_scores.toPandas()
with mlflow.start_run(run_name="llm_judge_eval_agent_v2"):
    mlflow.log_metric("avg_relevance", p.relevance.mean())
    mlflow.log_metric("avg_faithfulness", p.faithfulness.mean())
    mlflow.log_metric("pass_rate", (p.verdict == "pass").mean())
    mlflow.log_param("n_judged", len(p))
    mlflow.log_param("agent_change", "scope_refusal+filters+top5+grain+units")

print(f"agent v2 pass rate: {(p.verdict == 'pass').mean():.0%} over {len(p)} answers")
display(new_scores.join(spark.table("dbw_agentops.gold.agent_qa_log").select("trace_id", "answer"), "trace_id")
        .select("question", "relevance", "faithfulness", "verdict", "answer"))